# NER 中医药命名体识别

## 加载处理数据

In [1]:
categories = set()

def load_data(data_file):
    Data = {}
    with open (data_file, "rt", encoding="utf-8") as f:
        # 文本使用空行进行分割句子
        for idx, line in enumerate(f.read().split("\n\n")):
            if not line:
                break
            sentence, labels = "", []
            for i, item in enumerate(line.split("\n")):
                char, tag = item.split(" ")
                sentence += char
                if tag.startswith("B"):
                    labels.append([i, i, char, tag[2:]])   # Remove the B- or I-
                    categories.add(tag[2:])
                elif tag.startswith("I"):
                    labels[-1][1] = i
                    labels[-1][2] += char
            Data[idx] = {
                "sentence" : sentence,
                "labels" : labels
            }
    return Data

In [2]:
path = "dataset/ner_data/medical.train"
path_dev = "dataset/ner_data/medical.dev"

ds_train = load_data(path)
ds_dev = load_data(path_dev)

In [3]:
for idx, example in ds_train.items():
    print(idx, example)
    if idx >= 5:
        break
print("="*50)
for idx, example in ds_dev.items():
    print(idx, example)
    if idx >= 5:
        break

0 {'sentence': '现头昏口苦', 'labels': [[3, 4, '口苦', '临床表现']]}
1 {'sentence': '目的观察复方丁香开胃贴外敷神阙穴治疗慢性心功能不全伴功能性消化不良的临床疗效', 'labels': [[4, 10, '复方丁香开胃贴', '中医治疗'], [20, 32, '心功能不全伴功能性消化不良', '西医诊断']]}
2 {'sentence': '舒肝和胃消痞汤；功能性消化不良', 'labels': [[8, 14, '功能性消化不良', '西医诊断']]}
3 {'sentence': '患者３ａ前咯血，被诊断为肺结核，住院４０余天时出现腹痛，经治疗好转，但时有发作，坚持服抗痨药３ａ后，因腹痛基本缓解，肺结核治愈而停药', 'labels': [[5, 6, '咯血', '临床表现'], [12, 14, '肺结核', '西医诊断'], [58, 60, '肺结核', '西医诊断']]}
4 {'sentence': '治疗组采用复方蜥蜴散不同微粒组合剂（密点麻蜥、炙黄芪、焦乌梅、炒白芍、三七、半枝莲等）治疗', 'labels': [[5, 9, '复方蜥蜴散', '方剂'], [18, 21, '密点麻蜥', '中药'], [23, 25, '炙黄芪', '中药'], [27, 29, '焦乌梅', '中药'], [31, 33, '炒白芍', '中药'], [35, 36, '三七', '中药'], [38, 40, '半枝莲', '中药']]}
5 {'sentence': '经检查诊断为“缩窄性心包炎”', 'labels': [[7, 12, '缩窄性心包炎', '西医诊断']]}
0 {'sentence': '投活络效灵丹加味：当归、丹参各１５ｇ，生乳香、生没药各６ｇ，柴胡１２ｇ，白芍、黄芩、大黄各１０ｇ，蒲公英３０ｇ，甘草５ｇ', 'labels': [[1, 5, '活络效灵丹', '方剂'], [9, 10, '当归', '中药'], [12, 13, '丹参', '中药'], [19, 21, '生乳香', '中药'], [23, 25, '生没药', '中药'], [30, 31, '柴胡', '中药'], [39, 40, '黄芩', '中药'], [42, 43, '大黄

In [4]:
import json

def convert_to_sft_format(Data):
    sft_data = []
    for idx, sample in Data.items():
        sentence = sample["sentence"]
        labels = sample["labels"]

        # 构造output
        output = [
            {"text": text, "type": tag, "start": start, "end": end}
            for start, end, text, tag in labels
        ]
        prompt = """
        你是一个专业的生物医学信息抽取助手。请从用户提供的句子识别所有的实体，并标注它们的类型、起止位置，并输出标准 JSON。
        """
        sft_data.append({
            "instruction": prompt,
            "input": sentence,
            "output": json.dumps({"entities": output}, ensure_ascii=False)
        })
    return sft_data


In [5]:
from datasets import Dataset

ds_train_json = convert_to_sft_format(ds_train)
ds_valid_json = convert_to_sft_format(ds_dev)

train_tmp = Dataset.from_list(ds_train_json)
valid_tmp = Dataset.from_list(ds_valid_json)
train_tmp, valid_tmp

/root/miniconda3/envs/state3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(Dataset({
     features: ['instruction', 'input', 'output'],
     num_rows: 5259
 }),
 Dataset({
     features: ['instruction', 'input', 'output'],
     num_rows: 657
 }))

In [6]:
from transformers import AutoTokenizer
model_path = "./model/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [7]:
def process_func(example):
    max_length = 512

    system_prompt = """你是一个专业的医学信息抽取助手。
    请根据输入句子识别所有的实体，输出 JSON 格式，包含字段：
    text, type, start, end。

    示例：
    句子：气滞胃痛颗粒联合乳果糖治疗便秘型肠易激综合征临床研究
    输出：
    {
    "entities": [
        {"text": "气滞胃痛颗粒", "type": "中医治疗", "start": 0, "end": 5},
        {"text": "便秘型肠易激综合征", "type": "西医诊断", "start": 13, "end": 21},
    ]
    }
    """

    user_prompt = f"{example['instruction']}\n句子：{example['input']}"
    assistant_resp = example["output"]

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": assistant_resp},
    ]

    # 获取完整的 tokenized
    tokenized_full = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False, 
        max_length=max_length,
        truncation=True,
        return_tensors=None,
    )

    attention_mask = [1] * len(tokenized_full)
    
    # 获取 system + user 的 tokenized
    tokenized_prefix = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        tokenize=True,
        add_generation_prompt=True, # 添加 <|im_start|>assistant\n 的前缀
        return_tensors=None,
    )

    # Labels: mask 掉 prefix_len 之前的部分
    prefix_len = len(tokenized_prefix)
    labels = [-100] * min(prefix_len, len(tokenized_full)) + tokenized_full[min(prefix_len, len(tokenized_full)):]

    return {
        "input_ids": tokenized_full,
        "attention_mask": attention_mask,
        "labels": labels,
    }


In [8]:
train_data = train_tmp.map(process_func, remove_columns=train_tmp.column_names)
valid_data = valid_tmp.map(process_func, remove_columns=valid_tmp.column_names)

Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 657/657 [00:01<00:00, 506.54 examples/s]


In [9]:
print(tokenizer.decode(train_data[1]["input_ids"], skip_special_tokens=False))

<|im_start|>system
你是一个专业的医学信息抽取助手。
    请根据输入句子识别所有的实体，输出 JSON 格式，包含字段：
    text, type, start, end。

    示例：
    句子：气滞胃痛颗粒联合乳果糖治疗便秘型肠易激综合征临床研究
    输出：
    {
    "entities": [
        {"text": "气滞胃痛颗粒", "type": "中医治疗", "start": 0, "end": 5},
        {"text": "便秘型肠易激综合征", "type": "西医诊断", "start": 13, "end": 21},
    ]
    }
    <|im_end|>
<|im_start|>user

        你是一个专业的生物医学信息抽取助手。请从用户提供的句子识别所有的实体，并标注它们的类型、起止位置，并输出标准 JSON。
        
句子：目的观察复方丁香开胃贴外敷神阙穴治疗慢性心功能不全伴功能性消化不良的临床疗效<|im_end|>
<|im_start|>assistant
{"entities": [{"text": "复方丁香开胃贴", "type": "中医治疗", "start": 4, "end": 10}, {"text": "心功能不全伴功能性消化不良", "type": "西医诊断", "start": 20, "end": 32}]}<|im_end|>



## Args


In [10]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(model_path, device_map="cuda")
config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    inference_mode=False,  # 训练模式
    r=8,  # Lora 秩
    lora_alpha=32,  # Lora alaph，具体作用参见 Lora 原理
    lora_dropout=0.1,  # Dropout 比例
)

model = get_peft_model(model, config)

In [11]:
model.print_trainable_parameters()
model.enable_input_require_grads() 

trainable params: 8,544,256 || all params: 1,552,258,560 || trainable%: 0.5504


In [12]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq
import torch

args = TrainingArguments(
    output_dir="./output/Qwen2-ner_3",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    logging_steps=10,
    num_train_epochs=2,
    save_steps=200,
    learning_rate=5e-5,
    eval_strategy="steps",
    eval_steps=20,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    gradient_checkpointing=True,
    logging_dir="../tf-logs/qwen2_ner_3/rus",
    report_to="tensorboard",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_data,
    eval_dataset=valid_data,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)


In [13]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
20,0.251300,0.224478
40,0.181200,0.164934
60,0.147000,0.141937
80,0.133700,0.129627
100,0.119300,0.117650
120,0.112400,0.107284
140,0.105600,0.102464
160,0.099300,0.100565
180,0.092300,0.096039
200,0.089300,0.093103


TrainOutput(global_step=330, training_loss=0.1190750129295118, metrics={'train_runtime': 1994.6107, 'train_samples_per_second': 5.273, 'train_steps_per_second': 0.165, 'total_flos': 3.3986705434868736e+16, 'train_loss': 0.1190750129295118, 'epoch': 2.0})

In [17]:
lora_path='./qwen2.5_lora_ner'
trainer.model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)

('./qwen2.5_lora_ner/tokenizer_config.json',
 './qwen2.5_lora_ner/special_tokens_map.json',
 './qwen2.5_lora_ner/chat_template.jinja',
 './qwen2.5_lora_ner/vocab.json',
 './qwen2.5_lora_ner/merges.txt',
 './qwen2.5_lora_ner/added_tokens.json',
 './qwen2.5_lora_ner/tokenizer.json')

## 评估

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from peft import PeftModel

model_path = "./model/Qwen2.5-1.5B-Instruct"
lora_path='./qwen2.5_lora_ner'


local_lora_path = "./model/qwen2.5_lora_ner"
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# 加载模型
model = AutoModelForCausalLM.from_pretrained(model_name, 
                                            device_map="cuda",
                                            )

# 加载lora权重
model = PeftModel.from_pretrained(model, model_id=local_lora_path)

In [2]:
from seqeval.metrics import classification_report
import json

def span_to_bio_tags(sentence, entities): 
    # 假设我们按字符
    tokens = list(sentence)
    tags = ['O'] * len(tokens) # 初始化为 'O'

    for ent in entities:
        start = ent['start']
        end = ent['end'] # end 是实体结束位置的下一个索引
        ent_type = ent['type']

        if start < 0 or end > len(tokens):
            print(f"Warning: Entity span [{start}, {end}) is out of bounds for sentence '{sentence}'. Skipping.")
            continue

        tags[start] = f"B-{ent_type}"
        for i in range(start + 1, end):
            tags[i] = f"I-{ent_type}"

    return tags

In [3]:
categories = set()

def load_data(data_file):
    Data = {}
    with open (data_file, "rt", encoding="utf-8") as f:
        # 文本使用空行进行分割句子
        for idx, line in enumerate(f.read().split("\n\n")):
            if not line:
                break
            sentence, labels = "", []
            for i, item in enumerate(line.split("\n")):
                char, tag = item.split(" ")
                sentence += char
                if tag.startswith("B"):
                    labels.append([i, i, char, tag[2:]])   # Remove the B- or I-
                    categories.add(tag[2:])
                elif tag.startswith("I"):
                    labels[-1][1] = i
                    labels[-1][2] += char
            Data[idx] = {
                "sentence" : sentence,
                "labels" : labels
            }
    return Data

path_test = "dataset/ner_data/medical.test"

local_path_test = "state3/dataset/ner_data/medical.test"

ds_test = load_data(local_path_test)

In [4]:
for i, item in ds_test.items():
    print(item)
    if i >3:
        break

{'sentence': '药进１０帖，黄疸稍退，饮食稍增，精神稍振', 'labels': [[6, 7, '黄疸', '中医诊断']]}
{'sentence': '加味左金丸联合法莫替丁治疗胃食管反流病临床观察', 'labels': [[7, 10, '法莫替丁', '西医治疗'], [13, 18, '胃食管反流病', '西医诊断']]}
{'sentence': '“疏肝行气，调神解郁”推拿法结合西药治疗腹泻型ＩＢＳ的临床疗效', 'labels': [[1, 4, '疏肝行气', '中医治则'], [6, 9, '调神解郁', '中医治则'], [11, 12, '推拿', '中医治疗'], [20, 21, '腹泻', '临床表现']]}
{'sentence': '服上方７剂后腹痛渐减，大便日行一次已成形，续治一月诸症消失', 'labels': [[6, 7, '腹痛', '临床表现']]}
{'sentence': '方法采用随机数字表将１６０例功能性便秘患者随机分为２组，试验组与对照组各８０例，对照组予以乳果糖治疗，试验组则判断患者的中医证型，予以采用相应的中药汤剂联合中医特色疗法，疗程为８周', 'labels': [[14, 18, '功能性便秘', '西医诊断']]}


In [5]:
import json

def convert_to_sft_format(Data):
    sft_data = []
    for idx, sample in Data.items():
        sentence = sample["sentence"]
        labels = sample["labels"]

        # 构造output
        output = [
            {"text": text, "type": tag, "start": start, "end": end}
            for start, end, text, tag in labels
        ]
        prompt = """
        你是一个专业的生物医学信息抽取助手。请从用户提供的句子识别所有的实体，并标注它们的类型、起止位置，并输出标准 JSON。
        包含字段：entity_text, type, start, end。
        """
        sft_data.append({
            "instruction": prompt,
            "input": sentence,
            "output": json.dumps({"entities": output}, ensure_ascii=False)
        })
    return sft_data

test = convert_to_sft_format(ds_test)

In [6]:
from torch.utils.data import DataLoader
import torch

batch_data = []
for i in range(20):
    messages = [
        {"role": "system", "content": test[i]["instruction"]},
        {"role": "user", "content": test[i]["input"]},
    ]
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    batch_data.append(text)

batch_inputs = tokenizer(
    batch_data,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=1024,
).to("cuda")

with torch.no_grad():
    generated_ids = model.generate(
        **batch_inputs,
        max_new_tokens=512,
        do_sample=False,     
        temperature=0.0,     
        top_p=0.9,
    )

assistant_outputs = []
for resp in tokenizer.batch_decode(generated_ids, skip_special_tokens=True):
    # 提取 "assistant" 后的内容
    parts = resp.split("assistant")
    if len(parts) > 1:
        text = parts[-1].strip()
    else:
        text = resp.strip()
    assistant_outputs.append(text)

for i, text in enumerate(assistant_outputs):
    print(f"\n===== 样本 {i} =====")
    print(text)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.



===== 样本 0 =====
{"entity_text": "黄疸", "type": "临床表现", "start": 4, "end": 5}

===== 样本 1 =====
[{"entity_text": "加味左金丸", "type": "方剂", "start": 2, "end": 6}, {"entity_text": "法莫替丁", "type": "西医诊断", "start": 8, "end": 12}]

===== 样本 2 =====
{"entity_text": "疏肝行气", "type": "中医治则", "start": 3, "end": 6}

===== 样本 3 =====
{"entity_text": "腹痛", "type": "临床表现", "start": 5, "end": 6}

===== 样本 4 =====
[{"entity_text": "乳果糖", "type": "西医诊断", "start": 32, "end": 35}, {"entity_text": "中药汤剂", "type": "中医治疗", "start": 60, "end": 64}]

===== 样本 5 =====
{"entity_text": "失眠", "type": "临床表现", "start": 10, "end": 12}

===== 样本 6 =====
{"entity_text": "七方胃痛胶囊", "type": "中药", "start": 5, "end": 10}

===== 样本 7 =====
[{"entity_text": "肠易激综合征", "type": "疾病", "start": 0, "end": 6}, {"entity_text": "拔罐疗法", "type": "中医治疗", "start": 14, "end": 18}]

===== 样本 8 =====
{"entity_text": "枳术丸", "type": "方剂", "start": 5, "end": 7}

===== 样本 9 =====
[{"entity_text": "针刺", "type": "中医治疗", "start": 9, "end": 10}, {"ent

In [7]:
assistant_outputs

['{"entity_text": "黄疸", "type": "临床表现", "start": 4, "end": 5}',
 '[{"entity_text": "加味左金丸", "type": "方剂", "start": 2, "end": 6}, {"entity_text": "法莫替丁", "type": "西医诊断", "start": 8, "end": 12}]',
 '{"entity_text": "疏肝行气", "type": "中医治则", "start": 3, "end": 6}',
 '{"entity_text": "腹痛", "type": "临床表现", "start": 5, "end": 6}',
 '[{"entity_text": "乳果糖", "type": "西医诊断", "start": 32, "end": 35}, {"entity_text": "中药汤剂", "type": "中医治疗", "start": 60, "end": 64}]',
 '{"entity_text": "失眠", "type": "临床表现", "start": 10, "end": 12}',
 '{"entity_text": "七方胃痛胶囊", "type": "中药", "start": 5, "end": 10}',
 '[{"entity_text": "肠易激综合征", "type": "疾病", "start": 0, "end": 6}, {"entity_text": "拔罐疗法", "type": "中医治疗", "start": 14, "end": 18}]',
 '{"entity_text": "枳术丸", "type": "方剂", "start": 5, "end": 7}',
 '[{"entity_text": "针刺", "type": "中医治疗", "start": 9, "end": 10}, {"entity_text": "针刺", "type": "中医治疗", "start": 23, "end": 24}]',
 '{\n            "entity_text": "大便泻下",\n            "type": "临床表现",\n            

In [8]:
true_out = []
for i in range(20):
    true_out.append(test[i]["output"])
true_out

['{"entities": [{"text": "黄疸", "type": "中医诊断", "start": 6, "end": 7}]}',
 '{"entities": [{"text": "法莫替丁", "type": "西医治疗", "start": 7, "end": 10}, {"text": "胃食管反流病", "type": "西医诊断", "start": 13, "end": 18}]}',
 '{"entities": [{"text": "疏肝行气", "type": "中医治则", "start": 1, "end": 4}, {"text": "调神解郁", "type": "中医治则", "start": 6, "end": 9}, {"text": "推拿", "type": "中医治疗", "start": 11, "end": 12}, {"text": "腹泻", "type": "临床表现", "start": 20, "end": 21}]}',
 '{"entities": [{"text": "腹痛", "type": "临床表现", "start": 6, "end": 7}]}',
 '{"entities": [{"text": "功能性便秘", "type": "西医诊断", "start": 14, "end": 18}]}',
 '{"entities": [{"text": "头晕", "type": "临床表现", "start": 30, "end": 31}, {"text": "腹胀胁满", "type": "临床表现", "start": 48, "end": 51}, {"text": "大便溏薄", "type": "临床表现", "start": 57, "end": 60}, {"text": "舌苔薄黄腻", "type": "临床表现", "start": 78, "end": 82}, {"text": "脉细微弦", "type": "临床表现", "start": 88, "end": 91}]}',
 '{"entities": [{"text": "七方胃痛胶囊", "type": "中医治疗", "start": 5, "end": 10}, {"text": "肝郁脾虚

In [17]:
import json
import re

def normalize_entities(responses):
    results = []

    for resp in responses:
        text = resp.strip()

        # 尝试解析 JSON
        try:
            data = json.loads(text)
        except json.JSONDecodeError:
            # 有些模型输出 list 或单对象但没包在 {} 中
            if text.startswith("[") and text.endswith("]"):
                try:
                    data = json.loads(text)
                except:
                    continue
            elif text.startswith("{") and text.endswith("}"):
                try:
                    data = [json.loads(text)]
                except:
                    continue
            else:
                continue  # 无法解析的跳过

        # 转换为标准格式
        if isinstance(data, dict) and "entities" in data:
            # 已经是目标格式
            results.append(data)
        elif isinstance(data, dict):
            # 单个实体
            entity = {
                "text": data.get("entity_text") or data.get("text"),
                "type": data.get("type"),
                "start": data.get("start"),
                "end": data.get("end"),
            }
            results.append({"entities": [entity]})
        elif isinstance(data, list):
            entities = []
            for d in data:
                if not isinstance(d, dict):
                    continue
                entities.append({
                    "text": d.get("entity_text") or d.get("text"),
                    "type": d.get("type"),
                    "start": d.get("start"),
                    "end": d.get("end"),
                })
            results.append({"entities": entities})
        else:
            # 其他不合法格式
            results.append({"entities": []})

    return results


In [18]:
normalized = normalize_entities(assistant_outputs)

for item in normalized:
    print(item)

{'entities': [{'text': '黄疸', 'type': '临床表现', 'start': 4, 'end': 5}]}
{'entities': [{'text': '加味左金丸', 'type': '方剂', 'start': 2, 'end': 6}, {'text': '法莫替丁', 'type': '西医诊断', 'start': 8, 'end': 12}]}
{'entities': [{'text': '疏肝行气', 'type': '中医治则', 'start': 3, 'end': 6}]}
{'entities': [{'text': '腹痛', 'type': '临床表现', 'start': 5, 'end': 6}]}
{'entities': [{'text': '乳果糖', 'type': '西医诊断', 'start': 32, 'end': 35}, {'text': '中药汤剂', 'type': '中医治疗', 'start': 60, 'end': 64}]}
{'entities': [{'text': '失眠', 'type': '临床表现', 'start': 10, 'end': 12}]}
{'entities': [{'text': '七方胃痛胶囊', 'type': '中药', 'start': 5, 'end': 10}]}
{'entities': [{'text': '肠易激综合征', 'type': '疾病', 'start': 0, 'end': 6}, {'text': '拔罐疗法', 'type': '中医治疗', 'start': 14, 'end': 18}]}
{'entities': [{'text': '枳术丸', 'type': '方剂', 'start': 5, 'end': 7}]}
{'entities': [{'text': '针刺', 'type': '中医治疗', 'start': 9, 'end': 10}, {'text': '针刺', 'type': '中医治疗', 'start': 23, 'end': 24}]}
{'entities': [{'text': '胃气回复', 'type': '中医诊断', 'start': 29, 'end': 3

In [19]:
normalized, true_out

([{'entities': [{'text': '黄疸', 'type': '临床表现', 'start': 4, 'end': 5}]},
  {'entities': [{'text': '加味左金丸', 'type': '方剂', 'start': 2, 'end': 6},
    {'text': '法莫替丁', 'type': '西医诊断', 'start': 8, 'end': 12}]},
  {'entities': [{'text': '疏肝行气', 'type': '中医治则', 'start': 3, 'end': 6}]},
  {'entities': [{'text': '腹痛', 'type': '临床表现', 'start': 5, 'end': 6}]},
  {'entities': [{'text': '乳果糖', 'type': '西医诊断', 'start': 32, 'end': 35},
    {'text': '中药汤剂', 'type': '中医治疗', 'start': 60, 'end': 64}]},
  {'entities': [{'text': '失眠', 'type': '临床表现', 'start': 10, 'end': 12}]},
  {'entities': [{'text': '七方胃痛胶囊', 'type': '中药', 'start': 5, 'end': 10}]},
  {'entities': [{'text': '肠易激综合征', 'type': '疾病', 'start': 0, 'end': 6},
    {'text': '拔罐疗法', 'type': '中医治疗', 'start': 14, 'end': 18}]},
  {'entities': [{'text': '枳术丸', 'type': '方剂', 'start': 5, 'end': 7}]},
  {'entities': [{'text': '针刺', 'type': '中医治疗', 'start': 9, 'end': 10},
    {'text': '针刺', 'type': '中医治疗', 'start': 23, 'end': 24}]},
  {'entities': [{'text

In [20]:
import json
from collections import defaultdict

def calculate_entity_f1(predictions, ground_truths):
    """
    计算实体级别的 F1 分数
    predictions: 模型预测结果列表
    ground_truths: 真实标签列表（JSON 字符串需要先解析）
    """
    
    def parse_entities(data):
        """解析实体数据"""
        entities = []
        if isinstance(data, str):
            try:
                data = json.loads(data)
            except:
                return entities
        
        if isinstance(data, dict) and 'entities' in data:
            entities = data['entities']
        elif isinstance(data, list):
            entities = data
        
        # 标准化实体格式
        standardized = []
        for ent in entities:
            if isinstance(ent, dict):
                standardized.append({
                    'text': ent.get('text', ''),
                    'type': ent.get('type', ''),
                    'start': ent.get('start', 0),
                    'end': ent.get('end', 0)
                })
        return standardized
    
    total_tp = 0
    total_fp = 0
    total_fn = 0
    
    category_stats = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})
    
    for pred_data, true_data in zip(predictions, ground_truths):
        # 解析实体
        pred_entities = parse_entities(pred_data)
        true_entities = parse_entities(true_data)
        
        # 转换为集合以便比较
        true_set = set()
        pred_set = set()
        
        for ent in true_entities:
            key = (ent['start'], ent['end'], ent['text'], ent['type'])
            true_set.add(key)
            category_stats[ent['type']]['total_true'] = category_stats[ent['type']].get('total_true', 0) + 1
        
        for ent in pred_entities:
            key = (ent['start'], ent['end'], ent['text'], ent['type'])
            pred_set.add(key)
            category_stats[ent['type']]['total_pred'] = category_stats[ent['type']].get('total_pred', 0) + 1
        
        # 计算 TP, FP, FN
        tp = len(true_set & pred_set)
        fp = len(pred_set - true_set)
        fn = len(true_set - pred_set)
        
        total_tp += tp
        total_fp += fp
        total_fn += fn
        
        # 更新类别统计
        for key in true_set & pred_set:
            ent_type = key[3]
            category_stats[ent_type]['tp'] += 1
        
        for key in pred_set - true_set:
            ent_type = key[3]
            category_stats[ent_type]['fp'] += 1
        
        for key in true_set - pred_set:
            ent_type = key[3]
            category_stats[ent_type]['fn'] += 1
    
    # 计算总体指标
    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    # 计算每个类别的指标
    category_metrics = {}
    for cat, stats in category_stats.items():
        cat_tp = stats['tp']
        cat_fp = stats['fp']
        cat_fn = stats['fn']
        
        cat_precision = cat_tp / (cat_tp + cat_fp) if (cat_tp + cat_fp) > 0 else 0
        cat_recall = cat_tp / (cat_tp + cat_fn) if (cat_tp + cat_fn) > 0 else 0
        cat_f1 = 2 * cat_precision * cat_recall / (cat_precision + cat_recall) if (cat_precision + cat_recall) > 0 else 0
        
        category_metrics[cat] = {
            'precision': cat_precision,
            'recall': cat_recall,
            'f1': cat_f1,
            'support': cat_tp + cat_fn  # 真实实体数量
        }
    
    return {
        'overall': {
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'tp': total_tp,
            'fp': total_fp,
            'fn': total_fn
        },
        'by_category': category_metrics
    }

def print_evaluation_results(results):
    """打印评估结果"""
    overall = results['overall']
    by_category = results['by_category']
    
    print("=== 总体评估结果 ===")
    print(f"精确率 (Precision): {overall['precision']:.4f}")
    print(f"召回率 (Recall): {overall['recall']:.4f}")
    print(f"F1 分数: {overall['f1']:.4f}")
    print(f"TP: {overall['tp']}, FP: {overall['fp']}, FN: {overall['fn']}")
    
    print("\n=== 按类别评估 ===")
    for category, metrics in sorted(by_category.items()):
        print(f"{category:15} - P: {metrics['precision']:.4f}, R: {metrics['recall']:.4f}, F1: {metrics['f1']:.4f}, 支持数: {metrics['support']}")

# 执行评估
results = calculate_entity_f1(normalized, true_out)
print_evaluation_results(results)

=== 总体评估结果 ===
精确率 (Precision): 0.0000
召回率 (Recall): 0.0000
F1 分数: 0.0000
TP: 0, FP: 18, FN: 25

=== 按类别评估 ===
中医治则            - P: 0.0000, R: 0.0000, F1: 0.0000, 支持数: 2
中医治疗            - P: 0.0000, R: 0.0000, F1: 0.0000, 支持数: 3
中医证候            - P: 0.0000, R: 0.0000, F1: 0.0000, 支持数: 2
中医诊断            - P: 0.0000, R: 0.0000, F1: 0.0000, 支持数: 1
中药              - P: 0.0000, R: 0.0000, F1: 0.0000, 支持数: 1
临床表现            - P: 0.0000, R: 0.0000, F1: 0.0000, 支持数: 10
医疗              - P: 0.0000, R: 0.0000, F1: 0.0000, 支持数: 0
方剂              - P: 0.0000, R: 0.0000, F1: 0.0000, 支持数: 1
疾病              - P: 0.0000, R: 0.0000, F1: 0.0000, 支持数: 0
西医治疗            - P: 0.0000, R: 0.0000, F1: 0.0000, 支持数: 1
西医诊断            - P: 0.0000, R: 0.0000, F1: 0.0000, 支持数: 4


In [21]:
import json
from collections import defaultdict

def calculate_entity_f1(predictions, ground_truths, mode="lenient"):
    """
    predictions: 模型预测结果 (list of dict or str)
    ground_truths: 真实结果 (list of dict or str)
    mode: "strict" 或 "lenient"（宽松匹配）
    """
    def parse_entities(data):
        if isinstance(data, str):
            try:
                data = json.loads(data)
            except:
                return []
        if isinstance(data, dict):
            ents = data.get("entities", [])
        elif isinstance(data, list):
            ents = data
        else:
            ents = []
        return [
            {"text": e.get("text", ""), "type": e.get("type", ""), 
             "start": int(e.get("start", 0)), "end": int(e.get("end", 0))}
            for e in ents if isinstance(e, dict)
        ]
    
    def is_overlap(e1, e2):
        if e1["type"] != e2["type"]:
            return False
        return not (e1["end"] <= e2["start"] or e2["end"] <= e1["start"])

    total_tp, total_fp, total_fn = 0, 0, 0
    category_stats = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})

    for pred, truth in zip(predictions, ground_truths):
        pred_ents = parse_entities(pred)
        true_ents = parse_entities(truth)

        if mode == "strict":
            pred_keys = {(e["start"], e["end"], e["type"]) for e in pred_ents}
            true_keys = {(e["start"], e["end"], e["type"]) for e in true_ents}
            tp = len(pred_keys & true_keys)
            fp = len(pred_keys - true_keys)
            fn = len(true_keys - pred_keys)
            total_tp += tp
            total_fp += fp
            total_fn += fn
            for s, e, t in pred_keys & true_keys:
                category_stats[t]["tp"] += 1
            for s, e, t in pred_keys - true_keys:
                category_stats[t]["fp"] += 1
            for s, e, t in true_keys - pred_keys:
                category_stats[t]["fn"] += 1
        else:  # lenient
            matched_true = set()
            for pred_ent in pred_ents:
                matched = False
                for j, true_ent in enumerate(true_ents):
                    if j in matched_true:
                        continue
                    if is_overlap(pred_ent, true_ent):
                        total_tp += 1
                        category_stats[pred_ent["type"]]["tp"] += 1
                        matched_true.add(j)
                        matched = True
                        break
                if not matched:
                    total_fp += 1
                    category_stats[pred_ent["type"]]["fp"] += 1
            # 统计未匹配的真实实体
            for j, true_ent in enumerate(true_ents):
                if j not in matched_true:
                    total_fn += 1
                    category_stats[true_ent["type"]]["fn"] += 1

    precision = total_tp / (total_tp + total_fp) if total_tp + total_fp > 0 else 0
    recall = total_tp / (total_tp + total_fn) if total_tp + total_fn > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

    category_metrics = {}
    for t, s in category_stats.items():
        p = s["tp"] / (s["tp"] + s["fp"]) if s["tp"] + s["fp"] > 0 else 0
        r = s["tp"] / (s["tp"] + s["fn"]) if s["tp"] + s["fn"] > 0 else 0
        f = 2 * p * r / (p + r) if p + r > 0 else 0
        category_metrics[t] = {"precision": p, "recall": r, "f1": f, "support": s["tp"] + s["fn"]}

    return {
        "overall": {"precision": precision, "recall": recall, "f1": f1},
        "by_category": category_metrics
    }


In [23]:
results = calculate_entity_f1(normalized, true_out)
results

{'overall': {'precision': 0.1111111111111111,
  'recall': 0.08,
  'f1': 0.09302325581395349},
 'by_category': {'临床表现': {'precision': 0.0,
   'recall': 0.0,
   'f1': 0,
   'support': 10},
  '中医诊断': {'precision': 0.0, 'recall': 0.0, 'f1': 0, 'support': 1},
  '方剂': {'precision': 0.5,
   'recall': 1.0,
   'f1': 0.6666666666666666,
   'support': 1},
  '西医诊断': {'precision': 0.0, 'recall': 0.0, 'f1': 0, 'support': 4},
  '西医治疗': {'precision': 0, 'recall': 0.0, 'f1': 0, 'support': 1},
  '中医治则': {'precision': 1.0,
   'recall': 0.5,
   'f1': 0.6666666666666666,
   'support': 2},
  '中医治疗': {'precision': 0.0, 'recall': 0.0, 'f1': 0, 'support': 3},
  '中药': {'precision': 0.0, 'recall': 0.0, 'f1': 0, 'support': 1},
  '中医证候': {'precision': 0, 'recall': 0.0, 'f1': 0, 'support': 2},
  '疾病': {'precision': 0.0, 'recall': 0, 'f1': 0, 'support': 0},
  '医疗': {'precision': 0.0, 'recall': 0, 'f1': 0, 'support': 0}}}

In [30]:
messages = [
        {"role": "system", "content": test[1]["instruction"]},
        {"role": "user", "content": test[1]["input"]},
    ]

template = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(template, tokenizer(template, return_tensors="pt"))

tokened_ids = tokenizer(template, return_tensors="pt").to("cuda")

generated_ids = model.generate(tokened_ids.input_ids, max_new_tokens=512)

print(generated_ids)

print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

<|im_start|>system

        你是一个专业的生物医学信息抽取助手。请从用户提供的句子识别所有的实体，并标注它们的类型、起止位置，并输出标准 JSON。
        包含字段：entity_text, type, start, end。
        <|im_end|>
<|im_start|>user
加味左金丸联合法莫替丁治疗胃食管反流病临床观察<|im_end|>
<|im_start|>assistant
 {'input_ids': tensor([[151644,   8948,    271,    286,    220,  56568, 101909, 104715, 100206,
         104316,  27369, 118797, 110498,   1773,  14880,  45181,  20002, 103008,
         109949, 102450, 104152, 101565,  90395, 111066, 104017,   9370,  31905,
           5373,  71618,  81433,  81812,  90395,  66017, 100142,   4718,   8997,
            286,  94305,    227,  95312,  44931,   5122,   2996,   4326,     11,
            943,     11,   1191,     11,    835,   8997,    260, 151645,    198,
         151644,    872,    198,  20929,  99375,  77559,  34230, 106256, 101101,
          24339, 100707, 100290, 101142, 101899, 100518,  99450,  35551,  94443,
          88653,  99252, 104595, 104144, 151645,    198, 151644,  77091,    198]]), 'attention_mask': tensor([[1